HW_4: Score Differential, Opponent Adjusted Game Control, Opponent Adjusted Average Win Probability 

In [ ]:
!pip install "numpy>=2.0" pandas requests statsmodels great_tables adjustText fastparquet tabulate pyarrow sportsdataverse
!pip install nfl_data_py --no-deps
!pip install appdirs

In [2]:
import os
import tabulate
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nfl_data_py as nfl
import statsmodels.formula.api as smf
import pyarrow
import sportsdataverse as sdv
import polars as pl
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from great_tables import GT, md
from adjustText import adjust_text
from IPython.display import display, Markdown
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, log_loss
from scipy.special import expit

In [3]:
# ---------------------------------------------------------
# Load FBS play-by-play data for seasons (2021–2025)
# ---------------------------------------------------------
seasons = list(range(2021, 2026))

# Fetch Play-by-Play data

pbp = sdv.cfb.load_cfb_pbp(seasons=seasons).to_pandas()
schedule = sdv.cfb.load_cfb_schedule(seasons=seasons).to_pandas()

# Filter completed plays and compute duration between play events
pbp_clean = pbp[
    (pbp['start.TimeSecsRem'].notna()) &
    (pbp['homeScore'].notna()) &
    (pbp['awayScore'].notna()) &
    (pbp['homeTeamName'].notna()) &
    (pbp['awayTeamName'].notna())
].copy()

pbp_clean = pbp_clean.sort_values(['season', 'game_id', 'start.TimeSecsRem'], ascending=[True, True, False])

pbp_clean['home_lead'] = pbp_clean['homeScore'] - pbp_clean['awayScore']

# Elapsed seconds from previous play state
prev_seconds = pbp_clean.groupby('game_id')['start.TimeSecsRem'].shift(1).fillna(3600)
pbp_clean['play_duration'] = prev_seconds - pbp_clean['start.TimeSecsRem']
pbp_clean['play_duration'] = np.maximum(0, pbp_clean['play_duration'])

print("PBP shape:", pbp.shape)
print("Clean PBP shape:", pbp_clean.shape)
print(pbp_clean.head())

PBP shape: (779904, 499)
Clean PBP shape: (779904, 501)
      season    game_id  game_play_number  pos_team_id              pos_team  \
3491    2021  401281942                 1          333  Alabama Crimson Tide   
3582    2021  401281942                94         2390      Miami Hurricanes   
3492    2021  401281942                 2          333  Alabama Crimson Tide   
3583    2021  401281942                95         2390      Miami Hurricanes   
3584    2021  401281942                96         2390      Miami Hurricanes   

      def_pos_team_id          def_pos_team  pos_team_score  \
3491             2390      Miami Hurricanes               0   
3582              333  Alabama Crimson Tide               3   
3492             2390      Miami Hurricanes               0   
3583              333  Alabama Crimson Tide               3   
3584              333  Alabama Crimson Tide               3   

      def_pos_team_score  half  ...  two_pt_wp_diff  pointAfterAttempt.id  \
3491   

In [4]:
# -----------------------------------------------
# MODEL 1: Score Differential and Home/Away 
# -----------------------------------------------
# Clean for SRS 
games_clean = schedule[
    (schedule['completed'] == True) &
    (schedule['home_points'].notna()) &
    (schedule['away_points'].notna()) &
    (schedule['away_division'] == 'fbs') &
    (schedule['season_type'].isin(['regular']))
][['season', 'week', 'home_team', 'away_team', 'home_points', 'away_points']].sort_values(
    ['season', 'week']
).reset_index(drop=True)

In [5]:
# Fixed SRS Function
def calc_srs_scratch(df):
    # Reset index to guarantee alignment with 0..n_games range
    df = df.reset_index(drop=True)
    
    teams = np.unique(np.concatenate([
        df['home_team'].unique(),
        df['away_team'].unique()
    ]))
    n_games = len(df)
    
    X = pd.DataFrame(0.0, index=range(n_games), columns=teams)
    
    for i, row in df.iterrows():
        X.loc[i, row['home_team']] = 1.0
        X.loc[i, row['away_team']] = -1.0
        
    # Add home-field effect
    X['home_effect'] = 1.0
    
    margin = df['home_points'] - df['away_points']
    X = sm.add_constant(X)
    fit = sm.OLS(margin, X).fit()
    srs_vec = fit.params.fillna(0)

     # Keep only team ratings for the rankings
    team_params = srs_vec.drop('home_effect')
    srs_centered = team_params - team_params.mean()
    
    return pd.DataFrame({
        'team': srs_centered.index,
        'SRS': np.round(srs_centered.values, 2)
    })

In [6]:
def compute_season_srs(season_df):
    year = season_df['season'].iloc[0]
    
    srs_df = calc_srs_scratch(season_df)
    srs_df['season'] = year
    
    return srs_df.sort_values(by='SRS', ascending=False).reset_index(drop=True)

ratings_list = []

for year, season_df in games_clean.groupby('season'):
    ratings_list.append(compute_season_srs(season_df))

ratings_by_season = pd.concat(ratings_list, ignore_index=True)

In [7]:
# Top 10 and Bottom 10 SRS by Season
for yr in sorted(ratings_by_season['season'].unique()):

    season_ratings = (
        ratings_by_season[ratings_by_season['season'] == yr]
        .sort_values(by='SRS', ascending=False))

    top_10 = season_ratings.head(10).copy()
    bottom_10 = season_ratings.tail(10).sort_values(by='SRS').copy()

    top_10['Rank'] = range(1, 11)
    bottom_10['Rank'] = range(
        len(season_ratings) - 9,
        len(season_ratings) + 1)

    top_10 = top_10[['Rank', 'team', 'SRS']].rename(columns={
        'team': 'Team',
        'SRS': 'SRS Rating'
    })

    bottom_10 = bottom_10[['Rank', 'team', 'SRS']].rename(columns={
        'team': 'Team',
        'SRS': 'SRS Rating'
    })

    display(Markdown(f"### Season: {yr} — Top 10"))
    display(Markdown(top_10.to_markdown(index=False)))

    display(Markdown(f"### Season: {yr} — Bottom 10"))
    display(Markdown(bottom_10.to_markdown(index=False)))

### Season: 2021 — Top 10

|   Rank | Team           |   SRS Rating |
|-------:|:---------------|-------------:|
|      1 | Georgia        |        35.6  |
|      2 | Alabama        |        29.24 |
|      3 | Ohio State     |        28.64 |
|      4 | Michigan       |        25.79 |
|      5 | Notre Dame     |        21.68 |
|      6 | Oklahoma State |        19.93 |
|      7 | Utah           |        19.83 |
|      8 | Cincinnati     |        19.37 |
|      9 | Iowa State     |        17.17 |
|     10 | Wisconsin      |        16.58 |

### Season: 2021 — Bottom 10

|   Rank | Team                  |   SRS Rating |
|-------:|:----------------------|-------------:|
|    122 | Florida International |       -33.08 |
|    123 | Jacksonville State    |       -32.5  |
|    124 | Massachusetts         |       -30.7  |
|    125 | Akron                 |       -29.85 |
|    126 | Temple                |       -28.78 |
|    127 | UConn                 |       -28    |
|    128 | Southern Miss         |       -22.2  |
|    129 | New Mexico State      |       -21.59 |
|    130 | Arkansas State        |       -21.32 |
|    131 | Duke                  |       -20.7  |

### Season: 2022 — Top 10

...

|   Rank | Team             |   SRS Rating |
|-------:|:-----------------|-------------:|
|    125 | Kent State       |       -34.82 |
|    126 | Tulsa            |       -32.76 |
|    127 | Southern Miss    |       -28.98 |
|    128 | Middle Tennessee |       -26.83 |
|    129 | New Mexico State |       -26.7  |
|    130 | Kennesaw State   |       -26.29 |
|    131 | UTEP             |       -23.18 |
|    132 | Temple           |       -21.86 |
|    133 | Ball State       |       -21.02 |
|    134 | Central Michigan |       -20.96 |

### Season: 2025 — Top 10

|   Rank | Team       |   SRS Rating |
|-------:|:-----------|-------------:|
|      1 | Indiana    |        37.3  |
|      2 | Ohio State |        32.52 |
|      3 | Texas Tech |        31.92 |
|      4 | Oregon     |        30.15 |
|      5 | Notre Dame |        29.46 |
|      6 | Utah       |        25.59 |
|      7 | Miami      |        24.65 |
|      8 | Georgia    |        22.48 |
|      9 | USC        |        21.57 |
|     10 | Iowa       |        21.04 |

### Season: 2025 — Bottom 10

|   Rank | Team          |   SRS Rating |
|-------:|:--------------|-------------:|
|    127 | Massachusetts |       -37.83 |
|    128 | Sam Houston   |       -28.86 |
|    129 | UL Monroe     |       -25.58 |
|    130 | Charlotte     |       -24.62 |
|    131 | Georgia State |       -23.22 |
|    132 | Akron         |       -22.97 |
|    133 | Ball State    |       -22.56 |
|    134 | UTEP          |       -20.68 |
|    135 | Kent State    |       -20.27 |
|    136 | Rice          |       -19.25 |

QUESTIONS 2 (chronological) and 3 (random): Train/Test data 

In [8]:
# SRS — Chronological 70/30 Train/Test Split
train_list = []
test_list = []

for year, season_df in games_clean.groupby('season'):
    
    season_df = season_df.sort_values('week').reset_index(drop=True)
    
    split_point = int(len(season_df) * 0.70)
    
    train_list.append(season_df.iloc[:split_point])
    test_list.append(season_df.iloc[split_point:])

srs_train = pd.concat(train_list, ignore_index=True)
srs_test = pd.concat(test_list, ignore_index=True)

print("Training games:", len(srs_train))
print("Testing games:", len(srs_test))

Training games: 2610
Testing games: 1121


In [9]:
# Fit SRS model using chronological training data

teams = np.unique(np.concatenate([
    srs_train['home_team'].unique(),
    srs_train['away_team'].unique()
]))

X_train_srs = pd.DataFrame(
    0.0,
    index=range(len(srs_train)),
    columns=teams)

for i, (_, row) in enumerate(srs_train.iterrows()):
    X_train_srs.loc[i, row['home_team']] = 1.0
    X_train_srs.loc[i, row['away_team']] = -1.0

# Home-team effect
X_train_srs['home_effect'] = 1.0

# Actual score differential
y_train_srs = (
    srs_train['home_points'] -
    srs_train['away_points'])

# Fit model
srs_fit = sm.OLS(
    y_train_srs,
    X_train_srs).fit()

# Extract team ratings and home effect
team_params_srs = srs_fit.params.drop('home_effect')
home_effect_srs = srs_fit.params['home_effect']

print("Home effect:", round(home_effect_srs, 4))

Home effect: 2.5703


In [10]:
# Make SRS predictions on the chronological test data

srs_test['home_SRS'] = (
    srs_test['home_team']
    .map(team_params_srs)
    .fillna(0))

srs_test['away_SRS'] = (
    srs_test['away_team']
    .map(team_params_srs)
    .fillna(0))

srs_test['predicted_margin'] = (
    srs_test['home_SRS']
    - srs_test['away_SRS']
    + home_effect_srs)

srs_test['predicted_home_prob'] = expit(
    srs_test['predicted_margin'])

# Actual result
srs_test['actual_home_win'] = (
    srs_test['home_points'] >
    srs_test['away_points']).astype(int)

# Predicted result
srs_test['predicted_home_win'] = (
    srs_test['predicted_home_prob'] >= 0.5).astype(int)

In [11]:
accuracy_srs = accuracy_score(
    srs_test['actual_home_win'],
    srs_test['predicted_home_win'])

roc_auc_srs = roc_auc_score(
    srs_test['actual_home_win'],
    srs_test['predicted_home_prob'])
mse_srs = mean_squared_error(
    srs_test['actual_home_win'],
    srs_test['predicted_home_prob'])

logloss_srs = log_loss(
    srs_test['actual_home_win'],
    srs_test['predicted_home_prob'])

print("SRS Chronological 70/30")
print("Accuracy:", round(accuracy_srs, 4))
print("ROC-AUC:", round(roc_auc_srs, 4))
print("MSE:", round(mse_srs, 4))
print("Log Loss:", round(logloss_srs, 4))

SRS Chronological 70/30
Accuracy: 0.6815
ROC-AUC: 0.7316
MSE: 0.2876
Log Loss: 2.1714


In [12]:
# SRS — Random 70/30 Train/Test Split
train_list = []
test_list = []

for year, season_df in games_clean.groupby('season'):
    
    train, test = train_test_split(
        season_df,
        test_size=0.30,
        random_state=42
    )
    
    train_list.append(train)
    test_list.append(test)

srs_train_random = pd.concat(train_list, ignore_index=True)
srs_test_random = pd.concat(test_list, ignore_index=True)

print("Training games:", len(srs_train_random))
print("Testing games:", len(srs_test_random))

Training games: 2610
Testing games: 1121


In [13]:
# Fit SRS model using random training data

teams = np.unique(np.concatenate([
    srs_train_random['home_team'].unique(),
    srs_train_random['away_team'].unique()
]))

X_train_srs_random = pd.DataFrame(
    0.0,
    index=range(len(srs_train_random)),
    columns=teams)

for i, (_, row) in enumerate(srs_train_random.iterrows()):
    X_train_srs_random.loc[i, row['home_team']] = 1.0
    X_train_srs_random.loc[i, row['away_team']] = -1.0

# Home-team effect
X_train_srs_random['home_effect'] = 1.0

# Actual score differential
y_train_srs_random = (
    srs_train_random['home_points'] -
    srs_train_random['away_points'])

# Fit model
srs_fit_random = sm.OLS(
    y_train_srs_random,
    X_train_srs_random).fit()

# Extract team ratings and home effect
team_params_srs_random = (
    srs_fit_random.params.drop('home_effect'))

home_effect_srs_random = (
    srs_fit_random.params['home_effect'])

print("Home effect:", round(home_effect_srs_random, 4))

Home effect: 2.5112


In [14]:
# Make SRS predictions on the random test data

srs_test_random['home_SRS'] = (
    srs_test_random['home_team']
    .map(team_params_srs_random)
    .fillna(0))

srs_test_random['away_SRS'] = (
    srs_test_random['away_team']
    .map(team_params_srs_random)
    .fillna(0))

srs_test_random['predicted_margin'] = (
    srs_test_random['home_SRS']
    - srs_test_random['away_SRS']
    + home_effect_srs_random)

srs_test_random['predicted_home_prob'] = expit(
    srs_test_random['predicted_margin'])

# Actual result
srs_test_random['actual_home_win'] = (
    srs_test_random['home_points'] >
    srs_test_random['away_points']).astype(int)

# Predicted result
srs_test_random['predicted_home_win'] = (
    srs_test_random['predicted_home_prob'] >= 0.5).astype(int)

In [15]:
accuracy_srs_random = accuracy_score(
    srs_test_random['actual_home_win'],
    srs_test_random['predicted_home_win'])

roc_auc_srs_random = roc_auc_score(
    srs_test_random['actual_home_win'],
    srs_test_random['predicted_home_prob'])

mse_srs_random = mean_squared_error(
    srs_test_random['actual_home_win'],
    srs_test_random['predicted_home_prob'])

logloss_srs_random = log_loss(
    srs_test_random['actual_home_win'],
    srs_test_random['predicted_home_prob'])

print("SRS Random 70/30")
print("Accuracy:", round(accuracy_srs_random, 4))
print("ROC-AUC:", round(roc_auc_srs_random, 4))
print("MSE:", round(mse_srs_random, 4))
print("Log Loss:", round(logloss_srs_random, 4))

SRS Random 70/30
Accuracy: 0.678
ROC-AUC: 0.7472
MSE: 0.2895
Log Loss: 2.2138


In [17]:
# SRS Comparison

srs_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'ROC-AUC', 'MSE', 'Log Loss'],
    'Chronological 70/30': [
        accuracy_srs,
        roc_auc_srs,
        mse_srs,
        logloss_srs
    ],
    'Random 70/30': [
        accuracy_srs_random,
        roc_auc_srs_random,
        mse_srs_random,
        logloss_srs_random
    ]
})

print(srs_comparison.round(4).to_string(index=False))

  Metric  Chronological 70/30  Random 70/30
Accuracy               0.6815        0.6780
 ROC-AUC               0.7316        0.7472
     MSE               0.2876        0.2895
Log Loss               2.1714        2.2138


In [18]:
# ---------------------------------------------------------
# MODEL 2: Opponent Adjusted Game Control
# ---------------------------------------------------------
# Calculate time-weighted score differential per game for home and away teams
def calc_game_gc(group):
    total_time = group['play_duration'].sum()
    denom = total_time if total_time > 0 else 1.0
    home_gc = (group['home_lead'] * group['play_duration']).sum() / denom
    return pd.Series({
        'total_time': total_time,
        'home_gc': home_gc,
        'away_gc': -home_gc
    })

game_control_by_game = pbp_clean.groupby(['season', 'game_id', 'homeTeamName', 'awayTeamName'], group_keys=False).apply(calc_game_gc).reset_index()

# Reshape data into long team-game format
home_df = game_control_by_game[['season', 'game_id', 'homeTeamName', 'home_gc']].rename(columns={'homeTeamName': 'team', 'home_gc': 'game_control'})
away_df = game_control_by_game[['season', 'game_id', 'awayTeamName', 'away_gc']].rename(columns={'awayTeamName': 'team', 'away_gc': 'game_control'})

team_games = pd.concat([home_df, away_df], ignore_index=True)

# Aggregate across each season
season_game_control = team_games.groupby(['season', 'team']).agg(
    games_played=('game_control', 'count'),
    avg_game_control=('game_control', lambda x: np.round(x.mean(), 2))
).reset_index().sort_values(['season', 'avg_game_control'], ascending=[True, False])

In [19]:
# ---------------------------------------------------------
# Adjusted Game Control
# ---------------------------------------------------------
def calc_adj_gc_scratch(df):
    teams = np.unique(np.concatenate([df['homeTeamName'].unique(), df['awayTeamName'].unique()]))
    n_games = len(df)
    season = df['season'].iloc[0]
    
    # Construct design matrix: Home (+1), Away (-1)
    X = pd.DataFrame(0.0, index=range(n_games), columns=teams)
    
    for i, (_, row) in enumerate(df.iterrows()):
        X.loc[i, row['homeTeamName']] = 1.0
        X.loc[i, row['awayTeamName']] = -1.0
        
    margin = df['home_gc'].values
    
    # create a home effect
    X['home_effect'] = 1.0
    X_design = X
    
    #X_design=X   
    # Fit OLS model
    fit = sm.OLS(margin, X_design).fit()
    srs_vec = fit.params.fillna(0)
    
    # Center team parameters (home effect seperate)
    team_params = srs_vec.drop('home_effect')
    srs_centered = team_params - team_params.mean()

    home_effect = srs_vec['home_effect']
    
    return pd.DataFrame({
        'team': srs_centered.index,
        'season': season,
        'Adj_GC': np.round(srs_centered.values, 2)
    })

print("Game Control model completed.")
print("Number of games:", len(game_control_by_game))
print("Number of teams:", season_game_control['team'].nunique())
print(season_game_control.head())

Game Control model completed.
Number of games: 4508
Number of teams: 255
     season        team  games_played  avg_game_control
59     2021     Georgia            15              8.19
170    2021   Tennessee            13              7.53
3      2021     Alabama            15              7.07
132    2021  Ohio State            13              5.89
1      2021   Air Force            12              5.81


In [20]:
ratings_list = []
for yr, season_df in game_control_by_game.groupby('season'):
    ratings_list.append(calc_adj_gc_scratch(season_df))

ratings_by_season = pd.concat(ratings_list, ignore_index=True)

In [21]:
# ---------------------------------------------------------
# Top 10 and Bottom 10 by Season
# ---------------------------------------------------------
for yr in sorted(ratings_by_season['season'].unique()):
    
    season_ratings = ratings_by_season[
        ratings_by_season['season'] == yr
    ].sort_values('Adj_GC', ascending=False)

    top_10 = season_ratings.head(10).copy()
    bottom_10 = season_ratings.tail(10).sort_values('Adj_GC').copy()

    top_10['Rank'] = range(1, 11)
    bottom_10['Rank'] = range(
        len(season_ratings) - 9,
        len(season_ratings) + 1)

    top_10 = top_10[['Rank', 'team', 'Adj_GC']].rename(columns={
        'team': 'Team',
        'Adj_GC': 'Adjusted Game Control'
    })

    bottom_10 = bottom_10[['Rank', 'team', 'Adj_GC']].rename(columns={
        'team': 'Team',
        'Adj_GC': 'Adjusted Game Control'
    })

    display(Markdown(f"### Season: {yr} — Top 10"))
    display(Markdown(top_10.to_markdown(index=False)))

    display(Markdown(f"### Season: {yr} — Bottom 10"))
    display(Markdown(bottom_10.to_markdown(index=False)))

### Season: 2021 — Top 10

|   Rank | Team       |   Adjusted Game Control |
|-------:|:-----------|------------------------:|
|      1 | Georgia    |                   15.19 |
|      2 | Alabama    |                   14.22 |
|      3 | Tennessee  |                   11.8  |
|      4 | Ohio State |                   11    |
|      5 | Michigan   |                   10.03 |
|      6 | Wisconsin  |                    9.91 |
|      7 | Baylor     |                    9.57 |
|      8 | Ole Miss   |                    9.51 |
|      9 | Cincinnati |                    9.47 |
|     10 | Samford    |                    9.14 |

### Season: 2021 — Bottom 10

|   Rank | Team                     |   Adjusted Game Control |
|-------:|:-------------------------|------------------------:|
|    208 | Long Island University   |                  -31.77 |
|    209 | Southeast Missouri State |                  -28.21 |
|    210 | Eastern Illinois         |                  -22.8  |
|    211 | Abilene Christian        |                  -21.14 |
|    212 | Texas Southern           |                  -15.82 |
|    213 | Austin Peay              |                  -15.72 |
|    214 | Yale                     |                  -15.28 |
|    215 | Tennessee Tech           |                  -14.38 |
|    216 | South Carolina State     |                  -14.29 |
|    217 | Southern                 |                  -13.78 |

### Season: 2022 — Top 10

...

|   Rank | Team                |   Adjusted Game Control |
|-------:|:--------------------|------------------------:|
|    226 | Arkansas-Pine Bluff |                  -34.17 |
|    227 | Texas Southern      |                  -31.21 |
|    228 | Utah Tech           |                  -29.81 |
|    229 | Norfolk State       |                  -21.66 |
|    230 | Fordham             |                  -19.51 |
|    231 | Western Illinois    |                  -17.03 |
|    232 | Chattanooga         |                  -15.15 |
|    233 | Northwestern State  |                  -14.44 |
|    234 | Sacred Heart        |                  -13.75 |
|    235 | Merrimack           |                  -13.43 |

### Season: 2025 — Top 10

|   Rank | Team       |   Adjusted Game Control |
|-------:|:-----------|------------------------:|
|      1 | Indiana    |                   18.22 |
|      2 | Notre Dame |                   16.15 |
|      3 | Oregon     |                   13.6  |
|      4 | Iowa       |                   13.01 |
|      5 | Alabama    |                   12.65 |
|      6 | Texas Tech |                   12.54 |
|      7 | Ohio State |                   12.46 |
|      8 | Oklahoma   |                   11.87 |
|      9 | Miami      |                   11.15 |
|     10 | Ole Miss   |                   10.82 |

### Season: 2025 — Bottom 10

|   Rank | Team                 |   Adjusted Game Control |
|-------:|:---------------------|------------------------:|
|    227 | Tennessee State      |                  -38.11 |
|    228 | North Carolina A&T   |                  -34.02 |
|    229 | East Tennessee State |                  -26.56 |
|    230 | Houston Christian    |                  -24.25 |
|    231 | Grambling            |                  -20.45 |
|    232 | Central Connecticut  |                  -18.13 |
|    233 | Chattanooga          |                  -17.63 |
|    234 | Eastern Kentucky     |                  -15.75 |
|    235 | Northwestern State   |                  -15.02 |
|    236 | Colgate              |                  -14.29 |

In [22]:
#Correclation Average v Adjusted Game Control
all_df = season_game_control.merge(
    ratings_by_season, on=['season', 'team'], how='right'
)
all_df = all_df[all_df['team'] != '(Intercept)'].reset_index(drop=True)

QUESTIONS 2 (chronological) and 3 (random): Train/Test data

In [23]:
# Game Control — Chronological 70/30 Train/Test Split

train_list = []
test_list = []

for year, season_df in game_control_by_game.groupby('season'):
    
    season_df = season_df.sort_values('game_id').reset_index(drop=True)
    
    split_point = int(len(season_df) * 0.70)
    
    train_list.append(season_df.iloc[:split_point])
    test_list.append(season_df.iloc[split_point:])

gc_train = pd.concat(train_list, ignore_index=True)
gc_test = pd.concat(test_list, ignore_index=True)

print("Training games:", len(gc_train))
print("Testing games:", len(gc_test))

Training games: 3154
Testing games: 1354


In [24]:
# Fit Game Control model using training data only

teams = np.unique(np.concatenate([
    gc_train['homeTeamName'].unique(),
    gc_train['awayTeamName'].unique()
]))

X_train_gc = pd.DataFrame(
    0.0,
    index=range(len(gc_train)),
    columns=teams)

for i, (_, row) in enumerate(gc_train.iterrows()):
    X_train_gc.loc[i, row['homeTeamName']] = 1.0
    X_train_gc.loc[i, row['awayTeamName']] = -1.0

# Home-team effect
X_train_gc['home_effect'] = 1.0

# Game Control for the home team
y_train_gc = gc_train['home_gc']

# Fit the model
gc_fit = sm.OLS(
    y_train_gc,
    X_train_gc
).fit()

# Extract team ratings and home effect
team_params_gc = gc_fit.params.drop('home_effect')
home_effect_gc = gc_fit.params['home_effect']

print("Home effect:", round(home_effect_gc, 4))

Home effect: 0.8745


In [25]:
# Make Game Control predictions on the test data

gc_test['home_GC'] = (
    gc_test['homeTeamName']
    .map(team_params_gc)
    .fillna(0))

gc_test['away_GC'] = (
    gc_test['awayTeamName']
    .map(team_params_gc)
    .fillna(0))

gc_test['predicted_margin'] = (
    gc_test['home_GC']
    - gc_test['away_GC']
    + home_effect_gc)

gc_test['predicted_home_prob'] = expit(
    gc_test['predicted_margin'])

# Actual result
gc_test['actual_home_win'] = (
    gc_test['home_gc'] > gc_test['away_gc']).astype(int)

# Predicted result
gc_test['predicted_home_win'] = (
    gc_test['predicted_home_prob'] >= 0.5
).astype(int)

In [26]:
accuracy_gc = accuracy_score(
    gc_test['actual_home_win'],
    gc_test['predicted_home_win'])

roc_auc_gc = roc_auc_score(
    gc_test['actual_home_win'],
    gc_test['predicted_home_prob'])

mse_gc = mean_squared_error(
    gc_test['actual_home_win'],
    gc_test['predicted_home_prob'])

logloss_gc = log_loss(
    gc_test['actual_home_win'],
    gc_test['predicted_home_prob'])

print("Game Control Chronological 70/30")
print("Accuracy:", round(accuracy_gc, 4))
print("ROC-AUC:", round(roc_auc_gc, 4))
print("MSE:", round(mse_gc, 4))
print("Log Loss:", round(logloss_gc, 4))

Game Control Chronological 70/30
Accuracy: 0.6669
ROC-AUC: 0.7147
MSE: 0.2606
Log Loss: 0.9789


In [27]:
# Game Control — Random 70/30 Train/Test Split
train_list = []
test_list = []

for year, season_df in game_control_by_game.groupby('season'):
    
    train, test = train_test_split(
        season_df,
        test_size=0.30,
        random_state=42
    )
    
    train_list.append(train)
    test_list.append(test)

gc_train_random = pd.concat(train_list, ignore_index=True)
gc_test_random = pd.concat(test_list, ignore_index=True)

print("Training games:", len(gc_train_random))
print("Testing games:", len(gc_test_random))

Training games: 3154
Testing games: 1354


In [28]:
# Fit Game Control model using random training data

teams = np.unique(np.concatenate([
    gc_train_random['homeTeamName'].unique(),
    gc_train_random['awayTeamName'].unique()
]))

X_train_gc_random = pd.DataFrame(
    0.0,
    index=range(len(gc_train_random)),
    columns=teams)

for i, (_, row) in enumerate(gc_train_random.iterrows()):
    X_train_gc_random.loc[i, row['homeTeamName']] = 1.0
    X_train_gc_random.loc[i, row['awayTeamName']] = -1.0

# Home-team effect
X_train_gc_random['home_effect'] = 1.0

# Game Control for the home team
y_train_gc_random = gc_train_random['home_gc']

# Fit model
gc_fit_random = sm.OLS(
    y_train_gc_random,
    X_train_gc_random
).fit()

# Extract team ratings and home effect
team_params_gc_random = gc_fit_random.params.drop('home_effect')
home_effect_gc_random = gc_fit_random.params['home_effect']

print("Game Control random model fitted.")
print("Home effect:", round(home_effect_gc_random, 4))

Game Control random model fitted.
Home effect: 0.8771


In [29]:
# Make Game Control predictions on the random test data

gc_test_random['home_GC'] = (
    gc_test_random['homeTeamName']
    .map(team_params_gc_random)
    .fillna(0))

gc_test_random['away_GC'] = (
    gc_test_random['awayTeamName']
    .map(team_params_gc_random)
    .fillna(0))

gc_test_random['predicted_margin'] = (
    gc_test_random['home_GC']
    - gc_test_random['away_GC']
    + home_effect_gc_random)

gc_test_random['predicted_home_prob'] = expit(
    gc_test_random['predicted_margin'])

# Actual result
gc_test_random['actual_home_win'] = (
    gc_test_random['home_gc'] >
    gc_test_random['away_gc']).astype(int)

# Predicted result
gc_test_random['predicted_home_win'] = (
    gc_test_random['predicted_home_prob'] >= 0.5).astype(int)

In [30]:
accuracy_gc_random = accuracy_score(
    gc_test_random['actual_home_win'],
    gc_test_random['predicted_home_win'])

roc_auc_gc_random = roc_auc_score(
    gc_test_random['actual_home_win'],
    gc_test_random['predicted_home_prob'])

mse_gc_random = mean_squared_error(
    gc_test_random['actual_home_win'],
    gc_test_random['predicted_home_prob'])

logloss_gc_random = log_loss(
    gc_test_random['actual_home_win'],
    gc_test_random['predicted_home_prob'])

print("Game Control Random 70/30")
print("Accuracy:", round(accuracy_gc_random, 4))
print("ROC-AUC:", round(roc_auc_gc_random, 4))
print("MSE:", round(mse_gc_random, 4))
print("Log Loss:", round(logloss_gc_random, 4))

Game Control Random 70/30
Accuracy: 0.6869
ROC-AUC: 0.7567
MSE: 0.2371
Log Loss: 0.8776


In [31]:
# Game Control comparison

gc_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'ROC-AUC', 'MSE', 'Log Loss'],
    'Chronological 70/30': [
        accuracy_gc,
        roc_auc_gc,
        mse_gc,
        logloss_gc
    ],
    'Random 70/30': [
        accuracy_gc_random,
        roc_auc_gc_random,
        mse_gc_random,
        logloss_gc_random
    ]
})

print(gc_comparison.round(4).to_string(index=False))

  Metric  Chronological 70/30  Random 70/30
Accuracy               0.6669        0.6869
 ROC-AUC               0.7147        0.7567
     MSE               0.2606        0.2371
Log Loss               0.9789        0.8776


In [32]:
# ---------------------------------------------------------
# MODEL 3: Opponent Adjusted Average Win Probability (awp) 
# ---------------------------------------------------------

# Keep plays with valid win probability
awp_pbp = pbp_clean[
    (pbp_clean['home_wp_after'].notna()) &
    (pbp_clean['away_wp_after'].notna())
].copy()

print("AWP PBP shape:", awp_pbp.shape)
print(awp_pbp[['season', 'game_id', 'homeTeamName', 'awayTeamName',
               'home_wp_after', 'away_wp_after']].head())

AWP PBP shape: (779895, 501)
      season    game_id homeTeamName awayTeamName  home_wp_after  \
3491    2021  401281942        Miami      Alabama       0.090777   
3582    2021  401281942        Miami      Alabama       0.002116   
3492    2021  401281942        Miami      Alabama       0.092091   
3583    2021  401281942        Miami      Alabama       0.001919   
3584    2021  401281942        Miami      Alabama       0.001852   

      away_wp_after  
3491       0.909223  
3582       0.997884  
3492       0.907909  
3583       0.998081  
3584       0.998148  


In [33]:
# Average win probability for each team during each game
game_awp = awp_pbp.groupby(
    ['season', 'game_id', 'homeTeamName', 'awayTeamName']
).agg(
    home_awp=('home_wp_after', 'mean'),
    away_awp=('away_wp_after', 'mean')
).reset_index()

print("Game-level AWP shape:", game_awp.shape)
print(game_awp.head())

Game-level AWP shape: (4508, 6)
   season    game_id homeTeamName      awayTeamName  home_awp  away_awp
0    2021  401281942        Miami           Alabama  0.012115  0.987885
1    2021  401281944       Auburn             Akron  0.991117  0.008883
2    2021  401281945      Florida  Florida Atlantic  0.987368  0.012632
3    2021  401281946      Clemson           Georgia  0.333065  0.666935
4    2021  401281947     Kentucky         UL Monroe  0.984041  0.015959


In [34]:
# Create one row per team per game
home_awp = game_awp[
    ['season', 'game_id', 'homeTeamName', 'awayTeamName', 'home_awp']
].rename(columns={
    'homeTeamName': 'team',
    'awayTeamName': 'opponent',
    'home_awp': 'awp'
})

away_awp = game_awp[
    ['season', 'game_id', 'awayTeamName', 'homeTeamName', 'away_awp']
].rename(columns={
    'awayTeamName': 'team',
    'homeTeamName': 'opponent',
    'away_awp': 'awp'
})

team_awp = pd.concat([home_awp, away_awp], ignore_index=True)

print("Team-game AWP shape:", team_awp.shape)
print(team_awp.head())

Team-game AWP shape: (9016, 5)
   season    game_id      team          opponent       awp
0    2021  401281942     Miami           Alabama  0.012115
1    2021  401281944    Auburn             Akron  0.991117
2    2021  401281945   Florida  Florida Atlantic  0.987368
3    2021  401281946   Clemson           Georgia  0.333065
4    2021  401281947  Kentucky         UL Monroe  0.984041


In [35]:
# ---------------------------------------------------------
# Opponent-Adjusted AWP Model
# ---------------------------------------------------------

def calc_adj_awp_scratch(df):
    df = df.reset_index(drop=True)
    
    teams = np.unique(np.concatenate([
        df['homeTeamName'].unique(),
        df['awayTeamName'].unique()
    ]))
    
    n_games = len(df)
    season = df['season'].iloc[0]
    
    X = pd.DataFrame(0.0, index=range(n_games), columns=teams)
    
    for i, row in df.iterrows():
        X.loc[i, row['homeTeamName']] = 1.0
        X.loc[i, row['awayTeamName']] = -1.0
    
    # Home-team effect
    X['home_effect'] = 1.0
    
    # Home team's average win probability
    y = df['home_awp']
    
    fit = sm.OLS(y, X).fit()
    
    params = fit.params.fillna(0)
    
    # Remove home effect from team ratings
    team_params = params.drop('home_effect')
    
    # Center team ratings around zero
    team_params = team_params - team_params.mean()
    
    return pd.DataFrame({
        'team': team_params.index,
        'season': season,
        'AWP': np.round(team_params.values, 3)
    })

In [36]:
ratings_list = []

for year, season_df in game_awp.groupby('season'):
    ratings_list.append(calc_adj_awp_scratch(season_df))

awp_ratings_by_season = pd.concat(ratings_list, ignore_index=True)

print("AWP model completed.")
print("Number of team-season ratings:", len(awp_ratings_by_season))
print(awp_ratings_by_season.head())

AWP model completed.
Number of team-season ratings: 1145
                team  season    AWP
0  Abilene Christian    2021 -0.170
1          Air Force    2021  0.378
2              Akron    2021 -0.301
3            Alabama    2021  0.645
4      Alabama State    2021 -0.119


In [37]:
# Top 10 and Bottom 10 AWP by Season

for yr in sorted(awp_ratings_by_season['season'].unique()):

    season_ratings = (
        awp_ratings_by_season[awp_ratings_by_season['season'] == yr]
        .sort_values(by='AWP', ascending=False)
    )

    top_10 = season_ratings.head(10).copy()
    bottom_10 = season_ratings.tail(10).sort_values(by='AWP').copy()

    top_10['Rank'] = range(1, 11)
    bottom_10['Rank'] = range(
        len(season_ratings) - 9,
        len(season_ratings) + 1)

    top_10 = top_10[['Rank', 'team', 'AWP']].rename(columns={
        'team': 'Team',
        'AWP': 'AWP Rating'
    })

    bottom_10 = bottom_10[['Rank', 'team', 'AWP']].rename(columns={
        'team': 'Team',
        'AWP': 'AWP Rating'
    })

    display(Markdown(f"### Season: {yr} — Top 10"))
    display(Markdown(top_10.to_markdown(index=False)))

    display(Markdown(f"### Season: {yr} — Bottom 10"))
    display(Markdown(bottom_10.to_markdown(index=False)))

### Season: 2021 — Top 10

|   Rank | Team       |   AWP Rating |
|-------:|:-----------|-------------:|
|      1 | Georgia    |        0.678 |
|      2 | Alabama    |        0.645 |
|      3 | Ohio State |        0.552 |
|      4 | Michigan   |        0.545 |
|      5 | Oklahoma   |        0.485 |
|      6 | Wisconsin  |        0.485 |
|      7 | Cincinnati |        0.476 |
|      8 | Utah       |        0.473 |
|      9 | Pittsburgh |        0.456 |
|     10 | Baylor     |        0.436 |

### Season: 2021 — Bottom 10

|   Rank | Team               |   AWP Rating |
|-------:|:-------------------|-------------:|
|    208 | Bryant             |       -0.643 |
|    209 | Wagner             |       -0.618 |
|    210 | Central Arkansas   |       -0.587 |
|    211 | Gardner-Webb       |       -0.587 |
|    212 | Hampton            |       -0.574 |
|    213 | Florida A&M        |       -0.526 |
|    214 | Indiana State      |       -0.496 |
|    215 | Southern           |       -0.488 |
|    216 | North Carolina A&T |       -0.484 |
|    217 | Northwestern State |       -0.468 |

### Season: 2022 — Top 10

...

|   Rank | Team                |   AWP Rating |
|-------:|:--------------------|-------------:|
|    226 | Wagner              |       -0.621 |
|    227 | East Texas A&M      |       -0.517 |
|    228 | Northwestern State  |       -0.513 |
|    229 | Delaware State      |       -0.513 |
|    230 | Sacred Heart        |       -0.472 |
|    231 | Central Connecticut |       -0.456 |
|    232 | Bryant              |       -0.449 |
|    233 | Merrimack           |       -0.442 |
|    234 | Robert Morris       |       -0.436 |
|    235 | North Carolina A&T  |       -0.434 |

### Season: 2025 — Top 10

|   Rank | Team       |   AWP Rating |
|-------:|:-----------|-------------:|
|      1 | Indiana    |        0.691 |
|      2 | Ohio State |        0.676 |
|      3 | Notre Dame |        0.633 |
|      4 | Oregon     |        0.628 |
|      5 | Miami      |        0.601 |
|      6 | Texas Tech |        0.529 |
|      7 | Georgia    |        0.526 |
|      8 | Texas A&M  |        0.511 |
|      9 | Utah       |        0.509 |
|     10 | Ole Miss   |        0.509 |

### Season: 2025 — Bottom 10

|   Rank | Team             |   AWP Rating |
|-------:|:-----------------|-------------:|
|    227 | Bryant           |       -0.62  |
|    228 | UT Martin        |       -0.569 |
|    229 | Prairie View A&M |       -0.56  |
|    230 | Murray State     |       -0.558 |
|    231 | Saint Francis    |       -0.554 |
|    232 | Florida A&M      |       -0.51  |
|    233 | Lafayette        |       -0.493 |
|    234 | Delaware State   |       -0.491 |
|    235 | Bucknell         |       -0.468 |
|    236 | North Alabama    |       -0.46  |

QUESTIONS 2 (chronological) and 3 (random): Train/Test data

In [38]:
# AWP — Chronological 70/30 Train/Test Split
train_list = []
test_list = []

for year, season_df in game_awp.groupby('season'):
    
    season_df = season_df.sort_values('game_id').reset_index(drop=True)
    
    split_point = int(len(season_df) * 0.70)
    
    train_list.append(season_df.iloc[:split_point])
    test_list.append(season_df.iloc[split_point:])

awp_train = pd.concat(train_list, ignore_index=True)
awp_test = pd.concat(test_list, ignore_index=True)

print("Training games:", len(awp_train))
print("Testing games:", len(awp_test))

Training games: 3154
Testing games: 1354


In [39]:
# Fit AWP model using chronological training data

teams = np.unique(np.concatenate([
    awp_train['homeTeamName'].unique(),
    awp_train['awayTeamName'].unique()
]))

X_train_awp = pd.DataFrame(
    0.0,
    index=range(len(awp_train)),
    columns=teams)

for i, (_, row) in enumerate(awp_train.iterrows()):
    X_train_awp.loc[i, row['homeTeamName']] = 1.0
    X_train_awp.loc[i, row['awayTeamName']] = -1.0

# Home-team effect
X_train_awp['home_effect'] = 1.0

# Home team's average win probability
y_train_awp = awp_train['home_awp']

# Fit model
awp_fit = sm.OLS(
    y_train_awp,
    X_train_awp).fit()

# Extract team ratings and home effect
team_params_awp = awp_fit.params.drop('home_effect')
home_effect_awp = awp_fit.params['home_effect']

print("Home effect:", round(home_effect_awp, 4))

Home effect: 0.5587


In [40]:
# Make AWP predictions on the chronological test data

awp_test['home_AWP'] = (
    awp_test['homeTeamName']
    .map(team_params_awp)
    .fillna(0))

awp_test['away_AWP'] = (
    awp_test['awayTeamName']
    .map(team_params_awp)
    .fillna(0))

awp_test['predicted_margin'] = (
    awp_test['home_AWP']
    - awp_test['away_AWP']
    + home_effect_awp)

awp_test['predicted_home_prob'] = expit(
    awp_test['predicted_margin'])

# Actual result
awp_test['actual_home_win'] = (
    awp_test['home_awp'] >
    awp_test['away_awp']).astype(int)

# Predicted result
awp_test['predicted_home_win'] = (
    awp_test['predicted_home_prob'] >= 0.5).astype(int)

In [41]:
accuracy_awp = accuracy_score(
    awp_test['actual_home_win'],
    awp_test['predicted_home_win'])

roc_auc_awp = roc_auc_score(
    awp_test['actual_home_win'],
    awp_test['predicted_home_prob'])

mse_awp = mean_squared_error(
    awp_test['actual_home_win'],
    awp_test['predicted_home_prob'])

logloss_awp = log_loss(
    awp_test['actual_home_win'],
    awp_test['predicted_home_prob'])

print("AWP Chronological 70/30")
print("Accuracy:", round(accuracy_awp, 4))
print("ROC-AUC:", round(roc_auc_awp, 4))
print("MSE:", round(mse_awp, 4))
print("Log Loss:", round(logloss_awp, 4))

AWP Chronological 70/30
Accuracy: 0.6123
ROC-AUC: 0.7517
MSE: 0.2225
Log Loss: 0.6351


In [42]:
# AWP — Random 70/30 Train/Test Split

train_list = []
test_list = []

for year, season_df in game_awp.groupby('season'):
    
    train, test = train_test_split(
        season_df,
        test_size=0.30,
        random_state=42
    )
    
    train_list.append(train)
    test_list.append(test)

awp_train_random = pd.concat(train_list, ignore_index=True)
awp_test_random = pd.concat(test_list, ignore_index=True)

print("Training games:", len(awp_train_random))
print("Testing games:", len(awp_test_random))

Training games: 3154
Testing games: 1354


In [43]:
# Fit AWP model using random training data

teams = np.unique(np.concatenate([
    awp_train_random['homeTeamName'].unique(),
    awp_train_random['awayTeamName'].unique()
]))

X_train_awp_random = pd.DataFrame(
    0.0,
    index=range(len(awp_train_random)),
    columns=teams)

for i, (_, row) in enumerate(awp_train_random.iterrows()):
    X_train_awp_random.loc[i, row['homeTeamName']] = 1.0
    X_train_awp_random.loc[i, row['awayTeamName']] = -1.0

# Home-team effect
X_train_awp_random['home_effect'] = 1.0

# Home team's average win probability
y_train_awp_random = awp_train_random['home_awp']

# Fit model
awp_fit_random = sm.OLS(
    y_train_awp_random,
    X_train_awp_random).fit()

# Extract team ratings and home effect
team_params_awp_random = (
    awp_fit_random.params.drop('home_effect'))

home_effect_awp_random = (
    awp_fit_random.params['home_effect'])

print("Home effect:", round(home_effect_awp_random, 4))

Home effect: 0.5525


In [44]:
# Make AWP predictions on the random test data

awp_test_random['home_AWP'] = (
    awp_test_random['homeTeamName']
    .map(team_params_awp_random)
    .fillna(0))

awp_test_random['away_AWP'] = (
    awp_test_random['awayTeamName']
    .map(team_params_awp_random)
    .fillna(0))

awp_test_random['predicted_margin'] = (
    awp_test_random['home_AWP']
    - awp_test_random['away_AWP']
    + home_effect_awp_random)

awp_test_random['predicted_home_prob'] = expit(
    awp_test_random['predicted_margin'])

# Actual result
awp_test_random['actual_home_win'] = (
    awp_test_random['home_awp'] >
    awp_test_random['away_awp']).astype(int)

# Predicted result
awp_test_random['predicted_home_win'] = (
    awp_test_random['predicted_home_prob'] >= 0.5).astype(int)

In [45]:
accuracy_awp_random = accuracy_score(
    awp_test_random['actual_home_win'],
    awp_test_random['predicted_home_win'])

roc_auc_awp_random = roc_auc_score(
    awp_test_random['actual_home_win'],
    awp_test_random['predicted_home_prob'])

mse_awp_random = mean_squared_error(
    awp_test_random['actual_home_win'],
    awp_test_random['predicted_home_prob'])

logloss_awp_random = log_loss(
    awp_test_random['actual_home_win'],
    awp_test_random['predicted_home_prob'])

print("AWP Random 70/30")
print("Accuracy:", round(accuracy_awp_random, 4))
print("ROC-AUC:", round(roc_auc_awp_random, 4))
print("MSE:", round(mse_awp_random, 4))
print("Log Loss:", round(logloss_awp_random, 4))

AWP Random 70/30
Accuracy: 0.647
ROC-AUC: 0.836
MSE: 0.204
Log Loss: 0.5955


In [46]:
# AWP comparison

awp_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'ROC-AUC', 'MSE', 'Log Loss'],
    'Chronological 70/30': [
        accuracy_awp,
        roc_auc_awp,
        mse_awp,
        logloss_awp
    ],
    'Random 70/30': [
        accuracy_awp_random,
        roc_auc_awp_random,
        mse_awp_random,
        logloss_awp_random
    ]
})

print(gc_comparison.round(4).to_string(index=False))

  Metric  Chronological 70/30  Random 70/30
Accuracy               0.6669        0.6869
 ROC-AUC               0.7147        0.7567
     MSE               0.2606        0.2371
Log Loss               0.9789        0.8776


In [47]:
#compare all 3 
comparison = pd.DataFrame({
    'Model': [
        'SRS', 'SRS',
        'Game Control', 'Game Control',
        'AWP', 'AWP'
    ],
    'Split': [
        'Chronological 70/30', 'Random 70/30',
        'Chronological 70/30', 'Random 70/30',
        'Chronological 70/30', 'Random 70/30'
    ],
    'Accuracy': [
        accuracy_srs, accuracy_srs_random,
        accuracy_gc, accuracy_gc_random,
        accuracy_awp, accuracy_awp_random
    ],
    'ROC-AUC': [
        roc_auc_srs, roc_auc_srs_random,
        roc_auc_gc, roc_auc_gc_random,
        roc_auc_awp, roc_auc_awp_random
    ],
    'MSE': [
        mse_srs, mse_srs_random,
        mse_gc, mse_gc_random,
        mse_awp, mse_awp_random
    ],
    'Log Loss': [
        logloss_srs, logloss_srs_random,
        logloss_gc, logloss_gc_random,
        logloss_awp, logloss_awp_random
    ]
})

print(comparison.round(4).to_string(index=False))

       Model               Split  Accuracy  ROC-AUC    MSE  Log Loss
         SRS Chronological 70/30    0.6815   0.7316 0.2876    2.1714
         SRS        Random 70/30    0.6780   0.7472 0.2895    2.2138
Game Control Chronological 70/30    0.6669   0.7147 0.2606    0.9789
Game Control        Random 70/30    0.6869   0.7567 0.2371    0.8776
         AWP Chronological 70/30    0.6123   0.7517 0.2225    0.6351
         AWP        Random 70/30    0.6470   0.8360 0.2040    0.5955


QUESTION 2: Predictive 
I chose to split the training and testing data into a 70/30 split using chronological order. The first 70% of games are included in the training set, and the last 30% are included in the testing set. In order to test a model's predictive performance, I thought it would be best to split the data chronologically because the level of play in football can change throughout the season. Teams may improve, decline, gain momentum, or change their goals as the season progresses and playoff opportunities become clearer. Therefore, if we want to predict future games in terms of win probability, game control, and score differential, using earlier games to train the model and later games to test it provides a realistic measure of how well the models can predict future outcomes. I measured the models using Accuracy, ROC-AUC, MSE, and Log Loss to determine how well they performed at predicting future games. In terms of accuracy, the SRS model had the highest accuracy at 0.6815; however, all three models had relatively similar accuracy levels. I was most interested in how well the models could distinguish the target variable (ROC-AUC), how close the predicted probabilities were to the actual outcomes (MSE), and how well the predicted probabilities matched the actual outcomes (Log Loss). Once again, all three models produced relatively similar ROC-AUC and MSE values. However, Log Loss showed the clearest differences between the models and therefore provided the most useful metric for comparing their predicted probabilities. SRS produced a higher Log Loss of 2.17, Game Control produced a Log Loss of 0.9789, and AWP produced the lowest Log Loss at 0.6351. The higher the Log Loss, the more penalty the model places on incorrect predictions, so the lower score is better to interpret. With that, based on the metrics, the AWP model had the best predictive out-of-sample performance. Although it produced the lowest accuracy score, it had the highest ROC-AUC score, so it distinguished between the target variable the best. It also had the lowest MSE, so the predictive outcomes were the closest to the actual outcomes, and it incurred the lowest Log Loss.

QUESTION 3: Repeatability
Once again I chose to split the training and testing data into a 70/30 split, but this time doing it random. Doing a 70/30 split provides enough data to train the models while still leaving enough data to test their repeatability on unseen games. In order to test a model's repeatability performance, I thought it would be best to split the data randomly, since we don't want the results to depend on where they are at in the season. We just want to know if the models are able to repeat their performance. I measured the models using Accuracy, ROC-AUC, MSE, and Log Loss to determine how well they were able to do so. In terms of accuracy, the Game Control model had the highest accuracy at 0.6869; however, all three models had relatively similar accuracy levels. I was most interested in how well the models could distinguish the target variable (ROC-AUC), how close the predicted probabilities were to the actual outcomes (MSE), and how well the predicted probabilities matched the actual outcomes (Log Loss). Unlike the predictive model, the AWP model was a clear front runner across all of the metrics. AWP had the highest ROC-AUC (0.8360) by a significant amount, the lowest MSE (0.2040), and the lowest Log Loss (0.5955). Based on these results, the AWP model had the strongest repeatability performance. Although the Game Control model had the highest accuracy, AWP performed better across the other metrics, showing that its predictions were more consistent and repeatable on unseen games.

QUESTION 4: 

In [48]:
# Merge unadjusted AWP with adjusted AWP

awp_corr_data = team_awp.merge(
    awp_ratings_by_season,
    on=['season', 'team'],
    how='inner')

# Calculate correlations by season
awp_correlations = []

for year in sorted(awp_corr_data['season'].unique()):
    
    season_df = awp_corr_data[
        awp_corr_data['season'] == year
    ]
    
    correlation = season_df['awp'].corr(
        season_df['AWP'])
    
    awp_correlations.append({
        'Season': year,
        'Correlation': correlation
    })

# All seasons combined
awp_all = awp_corr_data['awp'].corr(
    awp_corr_data['AWP'])

awp_correlations.append({
    'Season': 'All Seasons',
    'Correlation': awp_all
})

awp_correlation_df = pd.DataFrame(awp_correlations)

print(awp_correlation_df.round(4).to_string(index=False))

     Season  Correlation
       2021       0.5334
       2022       0.5163
       2023       0.5114
       2024       0.5049
       2025       0.5037
All Seasons       0.5129


In [51]:
# Merge unadjusted Game Control with adjusted Game Control

gc_corr_data = season_game_control.merge(
    ratings_by_season,
    on=['season', 'team'], how='inner')

# Calculate correlations by season
gc_correlations = []

for year in sorted(gc_corr_data['season'].unique()):
    
    season_df = gc_corr_data[
        gc_corr_data['season'] == year
    ]
    
    correlation = season_df['avg_game_control'].corr(
        season_df['Adj_GC'])
    
    gc_correlations.append({
        'Season': year,
        'Correlation': correlation
    })

# All seasons combined
gc_all = gc_corr_data['avg_game_control'].corr(
    gc_corr_data['Adj_GC'])

gc_correlations.append({
    'Season': 'All Seasons',
    'Correlation': gc_all
})

gc_correlation_df = pd.DataFrame(gc_correlations)

print(gc_correlation_df.round(4).to_string(index=False))

     Season  Correlation
       2021       0.9090
       2022       0.9317
       2023       0.9317
       2024       0.9279
       2025       0.9210
All Seasons       0.9237


In [50]:
# Combine Game Control and AWP correlations

correlation_comparison = pd.DataFrame({
    'Season': gc_correlation_df['Season'],
    'Game Control Correlation': gc_correlation_df['Correlation'],
    'AWP Correlation': awp_correlation_df['Correlation']
})

print(correlation_comparison.round(4).to_string(index=False))

     Season  Game Control Correlation  AWP Correlation
       2021                    0.9090           0.5334
       2022                    0.9317           0.5163
       2023                    0.9317           0.5114
       2024                    0.9279           0.5049
       2025                    0.9210           0.5037
All Seasons                    0.9237           0.5129
